In [8]:
from pathlib import Path

import infostop
import polars as pl
import yaml
from pyproj import Transformer
from detect_stops import filter_pings, cluster_pings, assign_stop_ids, create_stop_table

In [2]:
with open("../config.copenhagen_sample.yaml") as fp:
    config = yaml.safe_load(fp)
with open("05_detect_stops_config.yaml") as fp:
    stop_detections_config = yaml.safe_load(fp)

In [3]:
PARTITION = 0  # valid between 0 and 255

In [4]:
# configure infostop
params = stop_detections_config["infostop"]

model = infostop.Infostop(
    r1=params["r1"],
    r2=params["r2"],
    min_staying_time=params["min_staying_time"],
    max_time_between=params["max_time_between"],
    min_size=params["min_size"],
    distance_metric=params["distance_metric"],
)

# set up transformer for reprojection
transformer = Transformer.from_crs(4326, config["crs"])

In [5]:
data = pl.read_parquet(
    f"{config['dirs']['output']}/{config['city']}/partitioned/{PARTITION:02x}/pings.parquet"
)

In [6]:
filtered = filter_pings(
    data, min_pings=stop_detections_config["filtering"]["min_pings_per_day"]
)
filtered = filtered.with_columns(
    pl.col("timestamp").dt.epoch(time_unit="s").alias("epoch")
)

In [9]:
try:
    clustered = pl.read_parquet(
        f"{config['dirs']['output']}/{config['city']}/clustered_pings/{PARTITION:02x}.parquet"
    )
except FileNotFoundError:
    # runs about 5m
    clustered = cluster_pings(filtered, model)
    clustered = assign_stop_ids(clustered)
    Path(f"{config['dirs']['output']}/{config['city']}/clustered_pings").mkdir(
        parents=True, exist_ok=True
    )
    clustered.write_parquet(
        f"{config['dirs']['output']}/{config['city']}/clustered_pings/{PARTITION:02x}.parquet"
    )


In [10]:
stops = create_stop_table(clustered, transformer)
Path(f"{config['dirs']['output']}/{config['city']}/stops").mkdir(
    parents=True, exist_ok=True
)
stops.write_parquet(
    f"{config['dirs']['output']}/{config['city']}/stops/{PARTITION:02x}.parquet"
)

In [11]:
stops.head(3)

device_id,stop_id,interval_start,interval_stop,interval_lenght,latitude_mean,longitude_mean,latitude_std,longitude_std,number_of_pings,place_label
i64,i32,"datetime[μs, Europe/Budapest]","datetime[μs, Europe/Budapest]",duration[μs],f64,f64,f64,f64,u32,i64
4,1,2025-04-07 06:03:13 CEST,2025-04-12 17:39:15 CEST,5d 11h 36m 2s,55.718058,12.465532,13.0,7.0,37,1
4,2,2025-04-12 18:04:34 CEST,2025-04-12 23:37:00 CEST,5h 32m 26s,55.686993,12.532105,7.0,9.0,162,2
4,3,2025-04-13 00:08:46 CEST,2025-04-13 01:53:55 CEST,1h 45m 9s,55.717978,12.46554,25.0,11.0,64,1


In [12]:
print(
    clustered.filter(pl.col.place_label > -1)
    .head(3)
    .to_pandas()
    .to_markdown(index=False)
)

| timestamp                 |   device_id | aid_type   |   latitude |   longitude |   place_label |   stop_id |
|:--------------------------|------------:|:-----------|-----------:|------------:|--------------:|----------:|
| 2025-04-07 06:03:13+02:00 |           4 | IDFA       |    55.7181 |     12.4656 |             1 |         1 |
| 2025-04-07 06:06:14+02:00 |           4 | IDFA       |    55.718  |     12.4656 |             1 |         1 |
| 2025-04-07 06:08:31+02:00 |           4 | IDFA       |    55.718  |     12.4656 |             1 |         1 |
